# Tomato Disease Progression Model

Model outputs only:
1. Current crop status: healthy or infected, and infected diseases
2. 24h infection trend: reduces, stable, or worse

## SECTION 0 - Run ID and Path Setup

In [1]:
# SECTION 0: Run ID and Centralized Path Setup
import os
import logging
import re
from datetime import datetime
from pathlib import Path

def find_project_root(start_path=None):
    """Find project root by looking for pyproject.toml, setup.py, or .git."""
    if start_path is None:
        start_path = os.path.abspath('.')

    current = Path(start_path)
    markers = ['pyproject.toml', 'setup.py', '.git', 'requirements.txt']

    for _ in range(10):
        if any((current / m).exists() for m in markers):
            return str(current)
        parent = current.parent
        if parent == current:
            break
        current = parent

    return str(Path.cwd().parent)

def is_valid_run_id(run_id_candidate):
    if not isinstance(run_id_candidate, str):
        return False
    return bool(re.match(r'^\d{8}_\d{6}$', run_id_candidate.strip()))

PROJECT_ROOT = find_project_root()
print(f'Project root    : {PROJECT_ROOT}')

RUN_ID = None
if ("RUN_ID" in globals() and RUN_ID is not None and isinstance(RUN_ID, str) and is_valid_run_id(RUN_ID)):
    print(f'Reusing RUN_ID from memory: {RUN_ID}')
else:
    MODEL_DIR = os.path.join(PROJECT_ROOT, 'src', 'agritwin_gh', 'models')
    artifacts_base = os.path.join(MODEL_DIR, 'artifacts')
    valid_run_ids = []

    if os.path.exists(artifacts_base):
        existing_runs = sorted([
            d for d in os.listdir(artifacts_base)
            if d.startswith('disease_progression_') and os.path.isdir(os.path.join(artifacts_base, d))
        ], reverse=True)

        for folder in existing_runs:
            candidate = folder.replace('disease_progression_', '')
            if is_valid_run_id(candidate):
                valid_run_ids.append(candidate)

        if valid_run_ids:
            RUN_ID = valid_run_ids[0]
            print(f'Detected and reusing existing RUN_ID: {RUN_ID}')
        else:
            RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
            print(f'Created new RUN_ID: {RUN_ID}')
    else:
        RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f'Created new RUN_ID: {RUN_ID}')

DATA_PATH = os.path.join(
    PROJECT_ROOT, 'data', 'processed', 'Disease Progression',
    'tomato_disease_progression_synthetic_hourly.csv'
)

MODEL_DIR = os.path.join(PROJECT_ROOT, 'src', 'agritwin_gh', 'models')
MODEL_PATH = os.path.join(MODEL_DIR, f'disease_progression_{RUN_ID}.pkl')

ARTIFACTS_DIR = os.path.join(MODEL_DIR, 'artifacts', f'disease_progression_{RUN_ID}')
PLOTS_DIR = ARTIFACTS_DIR
METRICS_DIR = ARTIFACTS_DIR
LOGS_DIR = ARTIFACTS_DIR
REPORTS_DIR = ARTIFACTS_DIR

for d in [MODEL_DIR, ARTIFACTS_DIR, PLOTS_DIR, METRICS_DIR, LOGS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

LOG_FILE = os.path.join(LOGS_DIR, f'run_{RUN_ID}.log')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger('disease_progression')
logger.info(f'Run started | run_id={RUN_ID}')

print(f'Using Run ID : {RUN_ID}')
print(f'Dataset      : {DATA_PATH}')
print(f'Model path   : {MODEL_PATH}')
print(f'Artifacts dir: {ARTIFACTS_DIR}')
print(f'Log file     : {LOG_FILE}')

2026-03-23 22:33:22,779 | INFO | Run started | run_id=20260323_192748


Project root    : c:\Users\pc\Documents\GitHub\AgriTwin-GH
Detected and reusing existing RUN_ID: 20260323_192748
Using Run ID : 20260323_192748
Dataset      : c:\Users\pc\Documents\GitHub\AgriTwin-GH\data\processed\Disease Progression\tomato_disease_progression_synthetic_hourly.csv
Model path   : c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\disease_progression_20260323_192748.pkl
Artifacts dir: c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260323_192748
Log file     : c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260323_192748\run_20260323_192748.log


## SECTION 1 - Setup

In [2]:
import json
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
import xgboost as xgb

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')

def save_fig(fig, name):
    path = os.path.join(PLOTS_DIR, f'{name}.png')
    fig.savefig(path, bbox_inches='tight', dpi=150)
    return path

print('Setup complete.')

Setup complete.


## SECTION 2 - Load and reshape disease dataset into crop-level realtime snapshots

In [3]:
assert os.path.exists(DATA_PATH), f'Dataset not found: {DATA_PATH}'
raw_df = pd.read_csv(DATA_PATH)
raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'])

print(f'Raw shape: {raw_df.shape}')
print('Disease labels:', sorted(raw_df['disease_name'].unique().tolist()))

DISEASES = sorted(raw_df['disease_name'].unique().tolist())

# Base per-hour crop conditions (same repeated across diseases at same timestamp/cycle).
base_cols = [
    'timestamp', 'cycle_id', 'cycle_label', 'season_label', 'stage_name', 'stage_index',
    'days_from_cycle_start', 'day_of_year', 'week_of_year', 'hour',
    'indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_CO2',
    'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy',
    'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy',
    'cumulative_gdd_like_index'
]
base_df = raw_df[base_cols].drop_duplicates(subset=['timestamp', 'cycle_id']).copy()

# Pivot disease-specific columns into wide format for realtime multi-disease modeling.
inf_pivot = raw_df.pivot_table(
    index=['timestamp', 'cycle_id'],
    columns='disease_name',
    values='current_infection_pct',
    aggfunc='mean'
).rename(columns={d: f'inf_{d}' for d in DISEASES}).reset_index()

risk_pivot = raw_df.pivot_table(
    index=['timestamp', 'cycle_id'],
    columns='disease_name',
    values='disease_risk_score',
    aggfunc='mean'
).rename(columns={d: f'risk_{d}' for d in DISEASES}).reset_index()

susc_pivot = raw_df.pivot_table(
    index=['timestamp', 'cycle_id'],
    columns='disease_name',
    values='stage_susceptibility_score',
    aggfunc='mean'
).rename(columns={d: f'susc_{d}' for d in DISEASES}).reset_index()

df = base_df.merge(inf_pivot, on=['timestamp', 'cycle_id'], how='left')
df = df.merge(risk_pivot, on=['timestamp', 'cycle_id'], how='left')
df = df.merge(susc_pivot, on=['timestamp', 'cycle_id'], how='left')

df.sort_values(['cycle_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

for d in DISEASES:
    df[f'inf_{d}'] = df[f'inf_{d}'].fillna(0.0)
    df[f'risk_{d}'] = df[f'risk_{d}'].fillna(0.0)
    df[f'susc_{d}'] = df[f'susc_{d}'].fillna(0.0)

df['total_infection_pct'] = df[[f'inf_{d}' for d in DISEASES]].sum(axis=1)

print(f'Realtime snapshot shape: {df.shape}')
display(df.head(3))

dataset_summary = {
    'raw_shape': [int(raw_df.shape[0]), int(raw_df.shape[1])],
    'snapshot_shape': [int(df.shape[0]), int(df.shape[1])],
    'diseases': DISEASES
}
with open(os.path.join(METRICS_DIR, 'dataset_summary.json'), 'w') as f:
    json.dump(dataset_summary, f, indent=2)

Raw shape: (54240, 39)
Disease labels: ['early_blight', 'late_blight', 'leaf_mold', 'powdery_mildew', 'spider_mites']
Realtime snapshot shape: (10848, 39)


,timestamp,cycle_id,cycle_label,season_label,stage_name,stage_index,days_from_cycle_start,day_of_year,week_of_year,hour,indoor_temp,indoor_humidity,indoor_air_velocity,indoor_CO2,solarradiation,day_night_flag,vpd,dew_point,leaf_wetness_proxy,temperature_rolling_mean_24h,humidity_rolling_mean_24h,vpd_proxy,cumulative_gdd_like_index,inf_early_blight,inf_late_blight,inf_leaf_mold,inf_powdery_mildew,inf_spider_mites,risk_early_blight,risk_late_blight,risk_leaf_mold,risk_powdery_mildew,risk_spider_mites,susc_early_blight,susc_late_blight,susc_leaf_mold,susc_powdery_mildew,susc_spider_mites,total_infection_pct
0,2024-07-01 00:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0000,183,27,0,29.002223,75.929629,2.41,440.0,0.0,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0.7918,0.0,0.0,0.0,0.0,0.0,0.35,0.1007,0.0817,0.4125,0.1893,0.35,0.45,0.3,0.5,0.4,0.0
1,2024-07-01 01:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0417,183,27,1,27.801924,70.830127,2.41,400.0,0.0,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,1.5335,0.0,0.0,0.0,0.0,0.0,0.35,0.0661,0.0646,0.4914,0.2009,0.35,0.45,0.3,0.5,0.4,0.0
2,2024-07-01 02:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0833,183,27,2,28.278680,70.035534,2.41,400.0,0.0,1.0,1.151145,22.285786,0.0,28.3609,72.2651,1.1511,2.2951,0.0,0.0,0.0,0.0,0.0,0.35,0.0548,0.0566,0.4923,0.2148,0.35,0.45,0.3,0.5,0.4,0.0


## SECTION 3 - Build outputs (targets) and realtime-safe features

In [4]:
INF_THRESHOLD = 0.5

TREND_TO_INT = {'reduces': 0, 'stable': 1, 'worse': 2}
INT_TO_TREND = {v: k for k, v in TREND_TO_INT.items()}

# Output 1 target: current disease infection status by disease + healthy flag.
for d in DISEASES:
    df[f'y_has_{d}'] = (df[f'inf_{d}'] >= INF_THRESHOLD).astype(int)

df['y_healthy'] = (df[[f'y_has_{d}' for d in DISEASES]].sum(axis=1) == 0).astype(int)

# Output 2 target: 24h trend (reduces / stable / worse).
# Use a composite delta so worsening is detectable in realistic scenarios.
df['max_infection_pct'] = df[[f'inf_{d}' for d in DISEASES]].max(axis=1)
df['total_infection_24h'] = df.groupby('cycle_id')['total_infection_pct'].shift(-24)
df['max_infection_24h'] = df.groupby('cycle_id')['max_infection_pct'].shift(-24)

df['delta_total_24h'] = df['total_infection_24h'] - df['total_infection_pct']
df['delta_max_24h'] = df['max_infection_24h'] - df['max_infection_pct']
df['delta_composite_24h'] = 0.6 * df['delta_max_24h'] + 0.4 * df['delta_total_24h']

def make_trend_label(delta, eps):
    if pd.isna(delta):
        return np.nan
    if delta > eps:
        return 'worse'
    if delta < -eps:
        return 'reduces'
    return 'stable'

# Adaptive epsilon to avoid class collapse and keep a meaningful stable region.
candidate_eps = [1.0, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05]
chosen_eps = candidate_eps[-1]
for eps in candidate_eps:
    tmp = df['delta_composite_24h'].apply(lambda x: make_trend_label(x, eps))
    counts = tmp.value_counts(dropna=True)
    if {'reduces', 'stable', 'worse'}.issubset(set(counts.index)):
        chosen_eps = eps
        break

TREND_EPS = chosen_eps
df['y_trend_24h'] = df['delta_composite_24h'].apply(lambda x: make_trend_label(x, TREND_EPS))
df['y_trend_24h_int'] = df['y_trend_24h'].map(TREND_TO_INT)

# Disease favorability features (realistic environmental risk proxies from documentation).
DISEASE_RULES = {
    'early_blight': {
        'temp_opt': (25.0, 32.0), 'temp_stress': (18.0, 38.0),
        'hum_opt': (55.0, 80.0), 'hum_stress': (40.0, 95.0),
        'vpd_opt': (0.7, 2.0), 'vpd_stress': (0.1, 3.0),
        'lw_bonus': 0.22,
    },
    'late_blight': {
        'temp_opt': (10.0, 22.0), 'temp_stress': (5.0, 28.0),
        'hum_opt': (85.0, 100.0), 'hum_stress': (60.0, 100.0),
        'vpd_opt': (0.0, 0.6), 'vpd_stress': (0.0, 1.5),
        'lw_bonus': 0.30,
    },
    'leaf_mold': {
        'temp_opt': (16.0, 24.0), 'temp_stress': (10.0, 30.0),
        'hum_opt': (82.0, 100.0), 'hum_stress': (60.0, 100.0),
        'vpd_opt': (0.0, 0.5), 'vpd_stress': (0.0, 1.5),
        'lw_bonus': 0.28,
    },
    'powdery_mildew': {
        'temp_opt': (20.0, 28.0), 'temp_stress': (12.0, 35.0),
        'hum_opt': (40.0, 70.0), 'hum_stress': (20.0, 95.0),
        'vpd_opt': (0.6, 1.8), 'vpd_stress': (0.1, 3.0),
        'lw_bonus': 0.00,
    },
    'spider_mites': {
        'temp_opt': (28.0, 38.0), 'temp_stress': (18.0, 45.0),
        'hum_opt': (28.0, 52.0), 'hum_stress': (15.0, 80.0),
        'vpd_opt': (2.0, 4.5), 'vpd_stress': (0.5, 5.5),
        'lw_bonus': -0.18,
    },
}

def piecewise_score(x, opt_low, opt_high, stress_low, stress_high):
    out = np.zeros_like(x, dtype=float)

    in_opt = (x >= opt_low) & (x <= opt_high)
    out[in_opt] = 1.0

    left = (x > stress_low) & (x < opt_low)
    if opt_low > stress_low:
        out[left] = (x[left] - stress_low) / (opt_low - stress_low)

    right = (x > opt_high) & (x < stress_high)
    if stress_high > opt_high:
        out[right] = (stress_high - x[right]) / (stress_high - opt_high)

    return np.clip(out, 0.0, 1.0)

for d in DISEASES:
    cfg = DISEASE_RULES[d]

    t = df['indoor_temp'].values
    h = df['indoor_humidity'].values
    v = df['vpd'].values
    lw = df['leaf_wetness_proxy'].values
    av = df['indoor_air_velocity'].values

    temp_s = piecewise_score(t, cfg['temp_opt'][0], cfg['temp_opt'][1], cfg['temp_stress'][0], cfg['temp_stress'][1])
    hum_s = piecewise_score(h, cfg['hum_opt'][0], cfg['hum_opt'][1], cfg['hum_stress'][0], cfg['hum_stress'][1])
    vpd_s = piecewise_score(v, cfg['vpd_opt'][0], cfg['vpd_opt'][1], cfg['vpd_stress'][0], cfg['vpd_stress'][1])

    fav = 0.45 * temp_s + 0.35 * hum_s + 0.20 * vpd_s
    fav = fav + cfg['lw_bonus'] * lw

    airflow_penalty = np.where(av > 2.3, 0.9, 1.0)
    fav = np.clip(fav * airflow_penalty, 0.0, 1.0)

    df[f'fav_{d}'] = fav
    df[f'fav_susc_{d}'] = fav * df[f'susc_{d}']

# Realtime-safe lag and rolling features: use only current and past values.
lag_cols = ['indoor_temp', 'indoor_humidity', 'vpd', 'solarradiation', 'total_infection_pct', 'max_infection_pct']
for c in lag_cols:
    for lag in [1, 3, 6, 12, 24]:
        df[f'{c}_lag_{lag}'] = df.groupby('cycle_id')[c].shift(lag)
    for w in [6, 24]:
        df[f'{c}_roll_mean_{w}'] = (
            df.groupby('cycle_id')[c]
            .rolling(w, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

# Add disease infection dynamics and favorability lags for stronger trend prediction.
for d in DISEASES:
    for lag in [1, 3, 6, 12, 24]:
        df[f'inf_{d}_lag_{lag}'] = df.groupby('cycle_id')[f'inf_{d}'].shift(lag)

    df[f'inf_{d}_delta_1h'] = df[f'inf_{d}'] - df[f'inf_{d}_lag_1']
    df[f'inf_{d}_delta_3h'] = df[f'inf_{d}'] - df[f'inf_{d}_lag_3']
    df[f'inf_{d}_delta_6h'] = df[f'inf_{d}'] - df[f'inf_{d}_lag_6']

    for lag in [1, 3, 6]:
        df[f'fav_{d}_lag_{lag}'] = df.groupby('cycle_id')[f'fav_{d}'].shift(lag)

feature_cols = [
    'stage_name', 'stage_index', 'cycle_label', 'season_label',
    'days_from_cycle_start', 'day_of_year', 'week_of_year', 'hour',
    'indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_CO2',
    'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy',
    'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h',
    'vpd_proxy', 'cumulative_gdd_like_index', 'total_infection_pct', 'max_infection_pct'
] + [f'inf_{d}' for d in DISEASES] + [f'risk_{d}' for d in DISEASES] + [f'susc_{d}' for d in DISEASES] + [f'fav_{d}' for d in DISEASES] + [f'fav_susc_{d}' for d in DISEASES] + [
    c for c in df.columns if c.endswith(tuple([f'_lag_{k}' for k in [1, 3, 6, 12, 24]]))
] + [
    c for c in df.columns if c.endswith('_roll_mean_6') or c.endswith('_roll_mean_24')
] + [
    c for c in df.columns if c.endswith('_delta_1h') or c.endswith('_delta_3h') or c.endswith('_delta_6h')
]

# Keep rows where trend target exists (needs t+24h).
model_df = df.dropna(subset=['y_trend_24h_int']).copy()
model_df[feature_cols] = model_df[feature_cols].bfill().fillna(0)

print(f'TREND_EPS chosen: {TREND_EPS}')
print(f'Modeling rows: {model_df.shape[0]}')
print('Trend distribution:')
print(model_df['y_trend_24h'].value_counts())

TREND_EPS chosen: 1.0
Modeling rows: 10752
Trend distribution:
y_trend_24h
worse      3822
reduces    3492
stable     3438
Name: count, dtype: int64


## SECTION 4 - Train/validation/test split by cycle (leakage-safe)

In [5]:
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

unique_cycles = np.array(sorted(model_df['cycle_id'].unique()))
np.random.shuffle(unique_cycles)

n = len(unique_cycles)
n_train = max(1, int(n * TRAIN_FRAC))
n_val = max(1, int(n * VAL_FRAC))
n_test = n - n_train - n_val
if n_test < 1:
    n_val -= 1
    n_test = 1

train_cycles = set(unique_cycles[:n_train])
val_cycles = set(unique_cycles[n_train:n_train + n_val])
test_cycles = set(unique_cycles[n_train + n_val:])

tr_df = model_df[model_df['cycle_id'].isin(train_cycles)].copy()
va_df = model_df[model_df['cycle_id'].isin(val_cycles)].copy()
te_df = model_df[model_df['cycle_id'].isin(test_cycles)].copy()

X_tr = tr_df[feature_cols].copy()
X_va = va_df[feature_cols].copy()
X_te = te_df[feature_cols].copy()

X_tr_enc = pd.get_dummies(X_tr, columns=['stage_name', 'cycle_label', 'season_label'], dummy_na=False)
X_va_enc = pd.get_dummies(X_va, columns=['stage_name', 'cycle_label', 'season_label'], dummy_na=False)
X_te_enc = pd.get_dummies(X_te, columns=['stage_name', 'cycle_label', 'season_label'], dummy_na=False)

all_cols = sorted(set(X_tr_enc.columns) | set(X_va_enc.columns) | set(X_te_enc.columns))
X_tr_enc = X_tr_enc.reindex(columns=all_cols, fill_value=0)
X_va_enc = X_va_enc.reindex(columns=all_cols, fill_value=0)
X_te_enc = X_te_enc.reindex(columns=all_cols, fill_value=0)

target_cols = [f'y_has_{d}' for d in DISEASES] + ['y_trend_24h_int']
y_tr = tr_df[target_cols].copy()
y_va = va_df[target_cols].copy()
y_te = te_df[target_cols].copy()

y_cur_tr = y_tr[[f'y_has_{d}' for d in DISEASES]].copy()
y_cur_va = y_va[[f'y_has_{d}' for d in DISEASES]].copy()
y_cur_te = y_te[[f'y_has_{d}' for d in DISEASES]].copy()

y_trend_tr = y_tr['y_trend_24h_int'].astype(int).copy()
y_trend_va = y_va['y_trend_24h_int'].astype(int).copy()
y_trend_te = y_te['y_trend_24h_int'].astype(int).copy()

print(f'Train rows: {len(X_tr_enc)} | Val rows: {len(X_va_enc)} | Test rows: {len(X_te_enc)}')
print(f'Encoded features: {X_tr_enc.shape[1]}')
print('Train trend distribution (int):')
print(y_trend_tr.value_counts().sort_index())

split_summary = {
    'train_cycles': sorted([str(x) for x in train_cycles]),
    'val_cycles': sorted([str(x) for x in val_cycles]),
    'test_cycles': sorted([str(x) for x in test_cycles]),
    'train_rows': int(len(X_tr_enc)),
    'val_rows': int(len(X_va_enc)),
    'test_rows': int(len(X_te_enc)),
    'n_features_encoded': int(X_tr_enc.shape[1]),
    'target_cols': target_cols,
    'trend_eps': float(TREND_EPS)
}
with open(os.path.join(METRICS_DIR, 'split_summary.json'), 'w') as f:
    json.dump(split_summary, f, indent=2)

Train rows: 5280 | Val rows: 2688 | Test rows: 2784
Encoded features: 156
Train trend distribution (int):
y_trend_24h_int
0    1732
1    1930
2    1618
Name: count, dtype: int64


## SECTION 5 - Train single deployable bundle (disease + trend heads)

In [6]:
# Train disease head (multi-label)
disease_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=800,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        class_weight='balanced_subsample',
        min_samples_leaf=2,
    )
)
disease_model.fit(X_tr_enc, y_cur_tr)

# Trend head A: XGBoost (multiclass) with class-balanced sample weights
trend_model_xgb = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=700,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=2,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    eval_metric='mlogloss',
)

trend_counts = y_trend_tr.value_counts().to_dict()
trend_weight_map = {k: len(y_trend_tr) / (3.0 * v) for k, v in trend_counts.items()}
trend_sample_weights = y_trend_tr.map(trend_weight_map).values
trend_model_xgb.fit(X_tr_enc, y_trend_tr, sample_weight=trend_sample_weights)

# Trend head B: RandomForest multiclass
trend_model_rf = RandomForestClassifier(
    n_estimators=900,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    class_weight='balanced_subsample',
    min_samples_leaf=2,
)
trend_model_rf.fit(X_tr_enc, y_trend_tr)


def aligned_trend_proba_from_xgb(model, X):
    """XGBoost predict_proba already returns class order [0,1,2]."""
    proba = model.predict_proba(X)
    if proba.shape[1] == 3:
        return proba
    aligned = np.zeros((len(X), 3), dtype=float)
    aligned[:, :proba.shape[1]] = proba
    return aligned


def aligned_trend_proba_from_rf(model, X):
    """Align RF probabilities to class order [0,1,2]."""
    raw = model.predict_proba(X)
    classes = model.classes_
    aligned = np.zeros((len(X), 3), dtype=float)
    for i, cls in enumerate(classes):
        aligned[:, int(cls)] = raw[:, i]
    return aligned


def positive_class_proba(estimator, X):
    """Return probability for class=1 for binary outputs, robust to single-class estimators."""
    proba_raw = estimator.predict_proba(X)
    classes = estimator.classes_
    if 1 in classes:
        idx = int(np.where(classes == 1)[0][0])
        return proba_raw[:, idx]
    return np.zeros(len(X), dtype=float)


def trend_predict_with_params(proba_aligned, m0, m1, m2, t0, t1, t2):
    """Class-aware 3-way decision rule with multipliers and class thresholds."""
    p0 = proba_aligned[:, 0] * m0
    p1 = proba_aligned[:, 1] * m1
    p2 = proba_aligned[:, 2] * m2

    s0 = p0 - t0
    s1 = p1 - t1
    s2 = p2 - t2
    scores = np.column_stack([s0, s1, s2])
    return np.argmax(scores, axis=1).astype(int)


# Tune disease thresholds on validation set
DISEASE_THRESHOLDS = {}
for i, d in enumerate(DISEASES):
    est = disease_model.estimators_[i]
    p_va = positive_class_proba(est, X_va_enc)
    y_va_bin = y_cur_va.iloc[:, i].values.astype(int)

    best_thr = 0.5
    best_f1 = -1.0
    for thr in np.arange(0.10, 0.91, 0.05):
        pred = (p_va >= thr).astype(int)
        f1v = f1_score(y_va_bin, pred, zero_division=0)
        if f1v > best_f1:
            best_f1 = f1v
            best_thr = float(np.round(thr, 2))
    DISEASE_THRESHOLDS[d] = best_thr


# Build blended trend probabilities and tune blend + class decision params
val_proba_xgb = aligned_trend_proba_from_xgb(trend_model_xgb, X_va_enc)
val_proba_rf = aligned_trend_proba_from_rf(trend_model_rf, X_va_enc)

best_cfg = None
best_score = -1.0
for blend_alpha in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    proba_blend = blend_alpha * val_proba_xgb + (1.0 - blend_alpha) * val_proba_rf

    for m0 in [0.9, 1.0, 1.1]:
        for m1 in [0.9, 1.0, 1.1]:
            for m2 in [1.0, 1.1, 1.2, 1.3, 1.4, 1.6]:
                for t0 in [0.20, 0.15, 0.10, 0.05, 0.00]:
                    for t1 in [0.20, 0.15, 0.10, 0.05, 0.00]:
                        for t2 in [0.20, 0.15, 0.10, 0.05, 0.00, -0.05]:
                            pred_va = trend_predict_with_params(proba_blend, m0, m1, m2, t0, t1, t2)

                            macro_f1 = f1_score(y_trend_va, pred_va, average='macro', zero_division=0)
                            r0 = recall_score((y_trend_va == 0).astype(int), (pred_va == 0).astype(int), zero_division=0)
                            r1 = recall_score((y_trend_va == 1).astype(int), (pred_va == 1).astype(int), zero_division=0)
                            r2 = recall_score((y_trend_va == 2).astype(int), (pred_va == 2).astype(int), zero_division=0)

                            if r0 < 0.50 or r1 < 0.50:
                                continue

                            score = 0.55 * macro_f1 + 0.20 * r2 + 0.15 * min(r0, r1, r2) + 0.10 * ((r0 + r1 + r2) / 3.0)

                            if score > best_score:
                                best_score = score
                                best_cfg = (
                                    float(blend_alpha),
                                    float(m0), float(m1), float(m2),
                                    float(t0), float(t1), float(t2),
                                )

if best_cfg is None:
    best_cfg = (0.5, 1.0, 1.0, 1.2, 0.10, 0.10, 0.10)

TREND_BLEND_ALPHA, TREND_M0, TREND_M1, TREND_M2, TREND_T0, TREND_T1, TREND_T2 = best_cfg

# Keep a reusable validation blend for the recalibration cell
val_trend_proba = TREND_BLEND_ALPHA * val_proba_xgb + (1.0 - TREND_BLEND_ALPHA) * val_proba_rf

print('Training complete: disease head + blended trend head')
print('Disease thresholds:', DISEASE_THRESHOLDS)
print(
    'Trend params -> '
    f'alpha={TREND_BLEND_ALPHA:.2f}, '
    f'm0={TREND_M0:.2f}, m1={TREND_M1:.2f}, m2={TREND_M2:.2f}, '
    f't0={TREND_T0:.2f}, t1={TREND_T1:.2f}, t2={TREND_T2:.2f}'
)

Training complete: disease head + blended trend head
Disease thresholds: {'early_blight': 0.3, 'late_blight': 0.35, 'leaf_mold': 0.45, 'powdery_mildew': 0.4, 'spider_mites': 0.4}
Trend params -> alpha=0.00, m0=0.90, m1=0.90, m2=1.20, t0=0.20, t1=0.05, t2=-0.05


In [7]:
# Post-training trend recalibration (fast, no retraining)
# Uses blended validation probabilities.

if 'val_trend_proba' not in globals():
    val_proba_xgb = aligned_trend_proba_from_xgb(trend_model_xgb, X_va_enc)
    val_proba_rf = aligned_trend_proba_from_rf(trend_model_rf, X_va_enc)
    val_trend_proba = TREND_BLEND_ALPHA * val_proba_xgb + (1.0 - TREND_BLEND_ALPHA) * val_proba_rf

best_cfg_recal = None
best_score_recal = -1.0

for m0 in [0.9, 1.0, 1.1]:
    for m1 in [0.8, 0.9, 1.0, 1.1]:
        for m2 in [1.1, 1.2, 1.3, 1.4, 1.6, 1.8, 2.0]:
            for t0 in [0.10, 0.05, 0.00]:
                for t1 in [0.15, 0.10, 0.05, 0.00]:
                    for t2 in [0.15, 0.10, 0.05, 0.00, -0.05]:
                        pred_va = trend_predict_with_params(val_trend_proba, m0, m1, m2, t0, t1, t2)

                        macro_f1 = f1_score(y_trend_va, pred_va, average='macro', zero_division=0)
                        r0 = recall_score((y_trend_va == 0).astype(int), (pred_va == 0).astype(int), zero_division=0)
                        r1 = recall_score((y_trend_va == 1).astype(int), (pred_va == 1).astype(int), zero_division=0)
                        r2 = recall_score((y_trend_va == 2).astype(int), (pred_va == 2).astype(int), zero_division=0)
                        f2 = f1_score((y_trend_va == 2).astype(int), (pred_va == 2).astype(int), zero_division=0)

                        if r0 < 0.50 or r1 < 0.50:
                            continue

                        score = 0.50 * macro_f1 + 0.25 * r2 + 0.15 * f2 + 0.10 * min(r0, r1, r2)

                        if score > best_score_recal:
                            best_score_recal = score
                            best_cfg_recal = (float(m0), float(m1), float(m2), float(t0), float(t1), float(t2))

if best_cfg_recal is not None:
    TREND_M0, TREND_M1, TREND_M2, TREND_T0, TREND_T1, TREND_T2 = best_cfg_recal

print('Recalibrated trend params:')
print(
    f'alpha={TREND_BLEND_ALPHA:.2f}, '
    f'm0={TREND_M0:.2f}, m1={TREND_M1:.2f}, m2={TREND_M2:.2f}, '
    f't0={TREND_T0:.2f}, t1={TREND_T1:.2f}, t2={TREND_T2:.2f}'
)

Recalibrated trend params:
alpha=0.00, m0=1.00, m1=1.10, m2=2.00, t0=0.10, t1=0.00, t2=-0.05


## SECTION 6 - Evaluation and artifact export

In [8]:
# Predictions from single deployable bundle with tuned decision thresholds
# Disease predictions using per-disease tuned probability thresholds.
pred_cur_te = np.zeros((len(X_te_enc), len(DISEASES)), dtype=int)
for i, d in enumerate(DISEASES):
    est = disease_model.estimators_[i]
    p_te = positive_class_proba(est, X_te_enc)
    pred_cur_te[:, i] = (p_te >= DISEASE_THRESHOLDS[d]).astype(int)

# Trend predictions from blended trend head using tuned class-aware params.
proba_te_xgb = aligned_trend_proba_from_xgb(trend_model_xgb, X_te_enc)
proba_te_rf = aligned_trend_proba_from_rf(trend_model_rf, X_te_enc)
trend_proba_te = TREND_BLEND_ALPHA * proba_te_xgb + (1.0 - TREND_BLEND_ALPHA) * proba_te_rf

pred_trend_te_int = trend_predict_with_params(
    trend_proba_te,
    TREND_M0,
    TREND_M1,
    TREND_M2,
    TREND_T0,
    TREND_T1,
    TREND_T2,
)

# Per-disease metrics for Output 1
disease_metrics_rows = []
for i, d in enumerate(DISEASES):
    yt = y_cur_te.iloc[:, i].values
    yp = pred_cur_te[:, i]
    disease_metrics_rows.append({
        'disease': d,
        'threshold': DISEASE_THRESHOLDS[d],
        'accuracy': float(accuracy_score(yt, yp)),
        'precision': float(precision_score(yt, yp, zero_division=0)),
        'recall': float(recall_score(yt, yp, zero_division=0)),
        'f1': float(f1_score(yt, yp, zero_division=0))
    })

disease_metrics_df = pd.DataFrame(disease_metrics_rows)
disease_metrics_df.to_csv(os.path.join(METRICS_DIR, 'current_disease_metrics.csv'), index=False)

# Healthy status derived from disease predictions.
y_healthy_te = (y_cur_te.sum(axis=1) == 0).astype(int).values
pred_healthy_te = (pred_cur_te.sum(axis=1) == 0).astype(int)
healthy_metrics = {
    'accuracy': float(accuracy_score(y_healthy_te, pred_healthy_te)),
    'precision': float(precision_score(y_healthy_te, pred_healthy_te, zero_division=0)),
    'recall': float(recall_score(y_healthy_te, pred_healthy_te, zero_division=0)),
    'f1': float(f1_score(y_healthy_te, pred_healthy_te, zero_division=0))
}

# Trend metrics for Output 2.
trend_labels_int = [0, 1, 2]
trend_target_names = [INT_TO_TREND[i] for i in trend_labels_int]
trend_report = classification_report(
    y_trend_te,
    pred_trend_te_int,
    labels=trend_labels_int,
    target_names=trend_target_names,
    output_dict=True,
    zero_division=0
)
trend_acc = float(accuracy_score(y_trend_te, pred_trend_te_int))
r0 = float(recall_score((y_trend_te == 0).astype(int), (pred_trend_te_int == 0).astype(int), zero_division=0))
r1 = float(recall_score((y_trend_te == 1).astype(int), (pred_trend_te_int == 1).astype(int), zero_division=0))
r2 = float(recall_score((y_trend_te == 2).astype(int), (pred_trend_te_int == 2).astype(int), zero_division=0))
trend_macro_f1 = float(f1_score(y_trend_te, pred_trend_te_int, average='macro', zero_division=0))
trend_balanced_acc = float((r0 + r1 + r2) / 3.0)
worse_recall = r2
worse_f1 = float(f1_score((y_trend_te == 2).astype(int), (pred_trend_te_int == 2).astype(int), zero_division=0))

metrics_bundle = {
    'run_id': RUN_ID,
    'healthy_metrics': healthy_metrics,
    'disease_thresholds': DISEASE_THRESHOLDS,
    'trend_accuracy': trend_acc,
    'trend_macro_f1': trend_macro_f1,
    'trend_balanced_accuracy': trend_balanced_acc,
    'trend_class_recalls': {
        'reduces': r0,
        'stable': r1,
        'worse': r2
    },
    'trend_worse_recall': worse_recall,
    'trend_worse_f1': worse_f1,
    'trend_tuning': {
        'blend_alpha_xgb': TREND_BLEND_ALPHA,
        'm0': TREND_M0,
        'm1': TREND_M1,
        'm2': TREND_M2,
        't0': TREND_T0,
        't1': TREND_T1,
        't2': TREND_T2,
    },
    'trend_report': trend_report
}
with open(os.path.join(METRICS_DIR, 'model_metrics.json'), 'w') as f:
    json.dump(metrics_bundle, f, indent=2)

# Save predictions table
pred_rows = te_df[['timestamp', 'cycle_id']].copy()
for i, d in enumerate(DISEASES):
    pred_rows[f'actual_has_{d}'] = y_cur_te.iloc[:, i].values
    pred_rows[f'pred_has_{d}'] = pred_cur_te[:, i]

pred_rows['actual_healthy'] = y_healthy_te
pred_rows['pred_healthy'] = pred_healthy_te
pred_rows['actual_trend_24h'] = y_trend_te.map(INT_TO_TREND).values
pred_rows['pred_trend_24h'] = pd.Series(pred_trend_te_int).map(INT_TO_TREND).values
pred_rows.to_csv(os.path.join(REPORTS_DIR, 'test_predictions.csv'), index=False)

# Plots
fig, ax = plt.subplots(figsize=(9, 4))
prev = y_cur_te.mean(axis=0).values
ax.bar(DISEASES, prev, color='tomato')
ax.set_title('Test-set Infection Prevalence by Disease')
ax.set_ylabel('Positive rate')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
print('Saved:', save_fig(fig, 'infection_prevalence_test'))
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
trend_counts = pd.Series(y_trend_te).map(INT_TO_TREND).value_counts().reindex(['reduces', 'stable', 'worse'], fill_value=0)
ax.bar(trend_counts.index, trend_counts.values, color=['seagreen', 'gray', 'crimson'])
ax.set_title('24h Trend Distribution (Test)')
ax.set_ylabel('Count')
plt.tight_layout()
print('Saved:', save_fig(fig, 'trend_distribution_test'))
plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_trend_te, pred_trend_te_int, labels=trend_labels_int)
ConfusionMatrixDisplay(cm, display_labels=trend_target_names).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Trend 24h Confusion Matrix')
plt.tight_layout()
print('Saved:', save_fig(fig, 'trend_confusion_matrix'))
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
tmp = disease_metrics_df.sort_values('f1')
ax.barh(tmp['disease'], tmp['f1'], color='steelblue')
ax.set_title('Per-disease F1 (Current Infection Detection)')
ax.set_xlabel('F1 score')
plt.tight_layout()
print('Saved:', save_fig(fig, 'per_disease_f1'))
plt.close(fig)

# Trend feature importance from blended heads (weighted sum)
imp_xgb = pd.Series(trend_model_xgb.feature_importances_, index=X_tr_enc.columns)
imp_rf = pd.Series(trend_model_rf.feature_importances_, index=X_tr_enc.columns)
feat_imp = (TREND_BLEND_ALPHA * imp_xgb + (1.0 - TREND_BLEND_ALPHA) * imp_rf).sort_values(ascending=False)
feat_imp.head(50).to_csv(os.path.join(METRICS_DIR, 'trend_feature_importance_top50.csv'))

fig, ax = plt.subplots(figsize=(9, 6))
top_n = 20
feat_imp.head(top_n).sort_values().plot(kind='barh', ax=ax, color='slateblue')
ax.set_title('Top-20 Feature Importances (24h Trend Output)')
ax.set_xlabel('Importance')
plt.tight_layout()
print('Saved:', save_fig(fig, 'trend_feature_importance_top20'))
plt.close(fig)

print('Healthy metrics:', healthy_metrics)
print('Trend accuracy      :', round(trend_acc, 4))
print('Trend macro-F1      :', round(trend_macro_f1, 4))
print('Trend balanced acc  :', round(trend_balanced_acc, 4))
print('Trend recalls       :', {'reduces': round(r0, 4), 'stable': round(r1, 4), 'worse': round(r2, 4)})
print('Trend params        :', {'alpha': TREND_BLEND_ALPHA, 'm0': TREND_M0, 'm1': TREND_M1, 'm2': TREND_M2, 't0': TREND_T0, 't1': TREND_T1, 't2': TREND_T2})

Saved: c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260323_192748\infection_prevalence_test.png
Saved: c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260323_192748\trend_distribution_test.png
Saved: c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260323_192748\trend_confusion_matrix.png
Saved: c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260323_192748\per_disease_f1.png
Saved: c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260323_192748\trend_feature_importance_top20.png
Healthy metrics: {'accuracy': 0.9996408045977011, 'precision': 1.0, 'recall': 0.9985590778097982, 'f1': 0.9992790194664743}
Trend accuracy      : 0.6085
Trend macro-F1      : 0.4963
Trend balanced acc  : 0.5667
Trend recalls       : {'reduces': 0.0075, 'stable': 0.7737, 'wors

## SECTION 7 - Realtime inference function (strict output format)

In [9]:
# Persist single deployable bundle (disease + blended trend heads) and schema
model_bundle = {
    'disease_model': disease_model,
    'trend_model_xgb': trend_model_xgb,
    'trend_model_rf': trend_model_rf,
    'trend_blend_alpha_xgb': TREND_BLEND_ALPHA,
    'trend_params': {
        'm0': TREND_M0,
        'm1': TREND_M1,
        'm2': TREND_M2,
        't0': TREND_T0,
        't1': TREND_T1,
        't2': TREND_T2,
    },
    'disease_thresholds': DISEASE_THRESHOLDS,
    'trend_to_int': TREND_TO_INT,
    'int_to_trend': INT_TO_TREND,
}
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model_bundle, f)

schema = {
    'diseases': DISEASES,
    'feature_cols': feature_cols,
    'encoded_feature_cols': X_tr_enc.columns.tolist(),
    'infection_threshold': INF_THRESHOLD,
    'disease_thresholds': DISEASE_THRESHOLDS,
    'trend_labels': ['reduces', 'stable', 'worse'],
    'trend_blend_alpha_xgb': TREND_BLEND_ALPHA,
    'trend_params': {
        'm0': TREND_M0,
        'm1': TREND_M1,
        'm2': TREND_M2,
        't0': TREND_T0,
        't1': TREND_T1,
        't2': TREND_T2,
    },
    'model_path': MODEL_PATH
}
with open(os.path.join(ARTIFACTS_DIR, 'inference_schema.json'), 'w') as f:
    json.dump(schema, f, indent=2)

def realtime_predict_from_snapshot(snapshot_row: pd.Series) -> dict:
    """
    Input: one crop-level snapshot row with the same feature schema.
    Output only required fields:
      - current_health: healthy/infected
      - diseases: list of active diseases (empty when healthy)
      - after_24h: reduces/stable/worse
    """
    x = snapshot_row[feature_cols].to_frame().T.copy()
    x[feature_cols] = x[feature_cols].fillna(0)
    x = pd.get_dummies(x, columns=['stage_name', 'cycle_label', 'season_label'], dummy_na=False)
    x = x.reindex(columns=schema['encoded_feature_cols'], fill_value=0)

    # Disease decisions with tuned per-disease thresholds
    disease_pred = []
    for i, d in enumerate(DISEASES):
        est = disease_model.estimators_[i]
        p = positive_class_proba(est, x)[0]
        disease_pred.append(int(p >= DISEASE_THRESHOLDS[d]))

    # Trend decision with blended trend heads + tuned class-aware params
    proba_xgb = aligned_trend_proba_from_xgb(trend_model_xgb, x)
    proba_rf = aligned_trend_proba_from_rf(trend_model_rf, x)
    proba_blend = TREND_BLEND_ALPHA * proba_xgb + (1.0 - TREND_BLEND_ALPHA) * proba_rf

    trend_pred_local = trend_predict_with_params(
        proba_blend,
        TREND_M0,
        TREND_M1,
        TREND_M2,
        TREND_T0,
        TREND_T1,
        TREND_T2,
    )[0]

    diseases = [d for i, d in enumerate(DISEASES) if int(disease_pred[i]) == 1]
    health = 'healthy' if len(diseases) == 0 else 'infected'

    return {
        'current_health': health,
        'diseases': diseases,
        'after_24h': INT_TO_TREND[int(trend_pred_local)]
    }

# Realtime scenario demo on test snapshots
demo_idx = np.random.choice(te_df.index, size=min(8, len(te_df)), replace=False)
demo_out = []
for idx in demo_idx:
    out = realtime_predict_from_snapshot(te_df.loc[idx])
    demo_out.append({
        'timestamp': str(te_df.loc[idx, 'timestamp']),
        'cycle_id': str(te_df.loc[idx, 'cycle_id']),
        **out
    })

demo_df = pd.DataFrame(demo_out)
display(demo_df)
demo_df.to_csv(os.path.join(REPORTS_DIR, 'realtime_demo_outputs.csv'), index=False)

print('Single model bundle + schema saved.')
print('Model path    :', MODEL_PATH)
print('Artifacts dir :', ARTIFACTS_DIR)

,timestamp,cycle_id,current_health,diseases,after_24h
0,2025-05-30 14:00:00,3,healthy,[],stable
1,2025-06-28 10:00:00,3,infected,"[powdery_mildew, spider_mites]",worse
2,2025-05-05 01:00:00,3,infected,"[powdery_mildew, spider_mites]",worse
3,2025-05-31 01:00:00,3,healthy,[],stable
4,2025-06-01 06:00:00,3,healthy,[],stable
5,2025-03-10 21:00:00,3,healthy,[],stable
6,2025-06-11 15:00:00,3,infected,[early_blight],worse
7,2025-03-18 12:00:00,3,infected,[early_blight],worse


Single model bundle + schema saved.
Model path    : c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\disease_progression_20260323_192748.pkl
Artifacts dir : c:\Users\pc\Documents\GitHub\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260323_192748


## SECTION 8 - Final run summary

In [10]:
summary = {
    'run_id': RUN_ID,
    'dataset_path': DATA_PATH,
    'artifacts_dir': ARTIFACTS_DIR,
    'models': {
        'combined_multitask_model': MODEL_PATH
    },
    'required_outputs': [
        'current_health + disease list',
        'after_24h trend (reduces/stable/worse)'
    ],
    'healthy_metrics': healthy_metrics,
    'disease_thresholds': DISEASE_THRESHOLDS,
    'trend_accuracy': trend_acc,
    'trend_macro_f1': trend_macro_f1,
    'trend_balanced_accuracy': trend_balanced_acc,
    'trend_class_recalls': {
        'reduces': r0,
        'stable': r1,
        'worse': r2
    },
    'trend_params': {
        'm0': TREND_M0,
        'm1': TREND_M1,
        'm2': TREND_M2,
        't0': TREND_T0,
        't1': TREND_T1,
        't2': TREND_T2
    }
}

with open(os.path.join(ARTIFACTS_DIR, 'run_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Run complete.')
print(json.dumps(summary, indent=2))

Run complete.
{
  "run_id": "20260323_192748",
  "dataset_path": "c:\\Users\\pc\\Documents\\GitHub\\AgriTwin-GH\\data\\processed\\Disease Progression\\tomato_disease_progression_synthetic_hourly.csv",
  "artifacts_dir": "c:\\Users\\pc\\Documents\\GitHub\\AgriTwin-GH\\src\\agritwin_gh\\models\\artifacts\\disease_progression_20260323_192748",
  "models": {
    "combined_multitask_model": "c:\\Users\\pc\\Documents\\GitHub\\AgriTwin-GH\\src\\agritwin_gh\\models\\disease_progression_20260323_192748.pkl"
  },
  "required_outputs": [
    "current_health + disease list",
    "after_24h trend (reduces/stable/worse)"
  ],
  "healthy_metrics": {
    "accuracy": 0.9996408045977011,
    "precision": 1.0,
    "recall": 0.9985590778097982,
    "f1": 0.9992790194664743
  },
  "disease_thresholds": {
    "early_blight": 0.3,
    "late_blight": 0.35,
    "leaf_mold": 0.45,
    "powdery_mildew": 0.4,
    "spider_mites": 0.4
  },
  "trend_accuracy": 0.6084770114942529,
  "trend_macro_f1": 0.49632916813134